In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

date_str = datetime.now().strftime("%Y-%m-%d")
s3_base_path = f"s3://openalex-snapshots/full/{date_str}"

ENTITIES = [
    {
        "name": "topics",
        "source_table": "openalex.common.topics_api",
        "snapshot_table": "openalex.common.openalex_topics_snapshot",
        "id_transform": lambda df: df.withColumn("id", F.concat(F.lit("https://openalex.org/T"), F.col("id"))),
        "array_columns": ["keywords", "siblings"],
    },
    {
        "name": "subfields",
        "source_table": "openalex.common.subfields_api",
        "snapshot_table": "openalex.common.openalex_subfields_snapshot",
        "id_transform": lambda df: df.withColumn("id", F.concat(F.lit("https://openalex.org/subfields/"), F.col("id"))),
        "array_columns": ["display_name_alternatives", "topics", "siblings"],
    },
    {
        "name": "fields",
        "source_table": "openalex.common.fields_api",
        "snapshot_table": "openalex.common.openalex_fields_snapshot",
        "id_transform": lambda df: df.withColumn("id", F.concat(F.lit("https://openalex.org/fields/"), F.col("id"))),
        "array_columns": ["display_name_alternatives", "subfields", "siblings"],
    },
    {
        "name": "domains",
        "source_table": "openalex.common.domains_api",
        "snapshot_table": "openalex.common.openalex_domains_snapshot",
        "id_transform": lambda df: df.withColumn("id", F.concat(F.lit("https://openalex.org/domains/"), F.col("id"))),
        "array_columns": ["display_name_alternatives", "fields", "siblings"],
    },
]

### Export each entity to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/{entity}/`.

In [0]:
date_str = get_snapshot_date()
for entity in ENTITIES:
    name = entity["name"]
    snapshot_table = entity["snapshot_table"]

    # Read source, apply id_transform if defined, coalesce null arrays
    src = spark.read.table(entity["source_table"])
    if "original_id" in src.columns:
        src = src.drop("original_id")
    id_transform = entity.get("id_transform")
    if id_transform:
        src = id_transform(src)
    from pyspark.sql import functions as F
    for col_name in entity.get("array_columns", []):
        src = src.withColumn(col_name, F.coalesce(F.col(col_name), F.array()))

    # Write the canonical snapshot table
    src.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(snapshot_table)

    # Export both formats via shared utility
    df = spark.read.table(snapshot_table)
    export_partitioned_all_formats(spark, dbutils, df, date_str, name, salt=False)

print("All entities exported.")